# WMT-Human — Preprocessing

This notebook handles data loading, cleaning, and preparation for the WMT-Human analysis.


In [22]:
import pandas as pd
import numpy as np
import re


## Load Raw Data


In [23]:
# Load raw data
df = pd.read_csv('../data/wmt-human_en_de_judged.csv')

print("=" * 50)
print("RAW DATA")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


RAW DATA
Shape: (9779, 16)
Columns: ['Unnamed: 0', 'id', 'source', 'reference', 'translation', 'mean_human_score', 'individual_human_scores', 'n_annotations', 'human_#1', 'human_#2', 'human_#3', 'GPT_as_a_judge', 'GPT-MINI_as_a_judge', 'MISTRAL_as_a_judge', 'MIXTRAL_as_a_judge', 'LLAMA_as_a_judge']


,Unnamed: 0,id,source,reference,translation,mean_human_score,individual_human_scores,n_annotations,human_#1,human_#2,human_#3,GPT_as_a_judge,GPT-MINI_as_a_judge,MISTRAL_as_a_judge,MIXTRAL_as_a_judge,LLAMA_as_a_judge
0,0,0,Michael Jackson wore tape on his nose to get f...,Ehemaliger Bodyguard berichtet: Michael Jackso...,"Michael Jackson trug Klebeband auf der Nase, u...",3.333333,"[4, 2, 4]",3,4,2,4,6,5,6,6 (Perfect Meaning and Grammar),4
1,1,1,Michael Jackson wore tape on his nose to get f...,Ehemaliger Bodyguard berichtet: Michael Jackso...,"Michael Jackson trug Klebeband auf der Nase, u...",4.333333,"[4, 5, 4]",3,4,5,4,6,5,6,6 (Perfect Meaning and Grammar),4
2,2,2,Michael Jackson wore tape on his nose to get f...,Ehemaliger Bodyguard berichtet: Michael Jackso...,"Michael Jackson trug Klebeband auf der Nase, u...",4.666667,"[5, 5, 4]",3,5,5,4,6,5,6,6 (Perfect Meaning and Grammar),4
3,3,3,Michael Jackson wore tape on his nose to get f...,Ehemaliger Bodyguard berichtet: Michael Jackso...,"Michael Jackson trug Klebeband auf der Nase, u...",4.000000,"[4, 4, 4]",3,4,4,4,6,5,6,6 (Perfect Meaning and Grammar),4
4,4,4,Michael Jackson wore tape on his nose to get f...,Ehemaliger Bodyguard berichtet: Michael Jackso...,"Michael Jackson trug Klebeband auf der Nase, u...",2.666667,"[4, 0, 4]",3,4,0,4,6,6,6,6 (Perfect Meaning and Grammar),4


## Data Cleaning


In [24]:
# Drop unnecessary columns
columns_to_drop = [
    'Unnamed: 0', 'id', 'source', 'reference', 'translation',
    'mean_human_score', 'individual_human_scores', 'n_annotations',
    'GPT-MINI_as_a_judge', 'MIXTRAL_as_a_judge',
]

df = df.drop(columns=columns_to_drop)

# rename MISTRAL -> Mistral, etc.
df = df.rename(columns={
    'MISTRAL_as_a_judge': 'Mistral_as_a_judge',
    'LLAMA_as_a_judge': 'Llama_as_a_judge',
    'GPT_as_a_judge': 'GPT-4o_as_a_judge',
})

print(f"Shape after dropping columns: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")
df.head()


Shape after dropping columns: (9779, 6)
Remaining columns: ['human_#1', 'human_#2', 'human_#3', 'GPT-4o_as_a_judge', 'Mistral_as_a_judge', 'Llama_as_a_judge']


,human_#1,human_#2,human_#3,GPT-4o_as_a_judge,Mistral_as_a_judge,Llama_as_a_judge
0,4,2,4,6,6,4
1,4,5,4,6,6,4
2,5,5,4,6,6,4
3,4,4,4,6,6,4
4,4,0,4,6,6,4


In [25]:
# Check unique values before cleaning
print("Before cleaning:")
print(f"human #1 unique values: {df['human_#1'].unique()}")
print(f"human #2 unique values: {df['human_#2'].unique()}")
print(f"human #3 unique values: {df['human_#3'].unique()}")
print()
print("Mistral unique values: ", df['Mistral_as_a_judge'].unique())
print("LLAMA unique values: ", df['Llama_as_a_judge'].unique())
print("GPT-4o unique values: ", df['GPT-4o_as_a_judge'].unique())


Before cleaning:
human #1 unique values: [4 5 3 6 2 1 0]
human #2 unique values: [2 5 4 0 6 3 1]
human #3 unique values: [4 1 2 5 3 6 0]

Mistral unique values:  ['6' '4' '5' '2' '6 (Perfect Meaning and Grammar)'
 '6 (The translation is perfect, as it is identical to the reference human translation.)'
 '0' 'Evaluation Error'
 '6 (The candidate translation is identical to the reference translation, so it receives a perfect score.)'
 '6 (The candidate translation is identical to the reference translation, so it is a perfect match.)']
LLAMA unique values:  ['4' '5' '3' 'Rating: 4' 'Rating: 5' '6' 'Evaluation Error' 'Rating: 3'
 '2']
GPT-4o unique values:  ['6' '5' '4' '3' '2' '0' 'Evaluation Error']


In [26]:
def extract_valid_rating(series: pd.Series) -> pd.Series:
    """
    Extract leading number from strings in a Series,
    keeping only values between 0 and 6 inclusive.
    Non-matching or out-of-range values are returned as NaN.

    Parameters
    ----------
    series : pd.Series

    Returns
    -------
    pd.Series
        Series of integers in [0, 6] or NaN.
    """
    pattern = re.compile(r'^\s*(\d+)')

    values = []
    for item in series:
        text = str(item)
        match = pattern.match(text)
        if match:
            value = int(match.group(1))
            if 0 <= value <= 6:
                values.append(value)
            else:
                values.append(np.nan)
        else:
            values.append(np.nan)

    return pd.Series(values, index=series.index)

# Apply cleaning to all columns
for col in df.columns:
    df[col] = extract_valid_rating(df[col])

print("After cleaning:")
print("Mistral unique values: ", df['Mistral_as_a_judge'].unique())
print("LLAMA unique values: ", df['Llama_as_a_judge'].unique())
print("GPT-4o unique values: ", df['GPT-4o_as_a_judge'].unique())


After cleaning:
Mistral unique values:  [ 6.  4.  5.  2.  0. nan]
LLAMA unique values:  [ 4.  5.  3. nan  6.  2.]
GPT-4o unique values:  [ 6.  5.  4.  3.  2.  0. nan]


In [27]:
# Cast to int where possible (keeps NaN as float)
df = df.astype(int, errors='ignore')

print(f"Final shape: {df.shape}")
print()
print("Final unique values:")
print("Mistral unique values: ", df['Mistral_as_a_judge'].unique())
print("LLAMA unique values: ", df['Llama_as_a_judge'].unique())
print("GPT-4o unique values: ", df['GPT-4o_as_a_judge'].unique())
print()
print("Data types:")
print(df.dtypes)


Final shape: (9779, 6)

Final unique values:
Mistral unique values:  [ 6.  4.  5.  2.  0. nan]
LLAMA unique values:  [ 4.  5.  3. nan  6.  2.]
GPT-4o unique values:  [ 6.  5.  4.  3.  2.  0. nan]

Data types:
human_#1                int64
human_#2                int64
human_#3                int64
GPT-4o_as_a_judge     float64
Mistral_as_a_judge    float64
Llama_as_a_judge      float64
dtype: object


## Save Cleaned Data


In [28]:
# Save cleaned dataset
df.to_csv("../data/wmt-human_en_de_judged_cleaned.csv", index=False)
print(f"✅ Saved cleaned dataset to ../data/wmt-human_en_de_judged_cleaned.csv")
print(f"   Shape: {df.shape}")

print()
print("=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print()
print("Output file:")
print("  - ../data/wmt-human_en_de_judged_cleaned.csv")


✅ Saved cleaned dataset to ../data/wmt-human_en_de_judged_cleaned.csv
   Shape: (9779, 6)

PREPROCESSING COMPLETE

Output file:
  - ../data/wmt-human_en_de_judged_cleaned.csv
